In [ ]:
/**
 * =====================================================
 *  Google Apps Script - ESP32 PIR Data Receiver
 * =====================================================
 * 
 * CARA SETUP:
 *  1. Buka Google Spreadsheet baru
 *  2. Klik menu: Extensions → Apps Script
 *  3. Hapus kode default, paste seluruh kode ini
 *  4. Klik "Save" (Ctrl+S)
 *  5. Klik "Deploy" → "New Deployment"
 *  6. Pilih Type: "Web App"
 *  7. Execute as: "Me"
 *  8. Who has access: "Anyone"
 *  9. Klik "Deploy", izinkan akses, copy Web App URL
 * 10. Ambil bagian ID dari URL:
 *     https://script.google.com/macros/s/[SCRIPT_ID]/exec
 * 11. Paste SCRIPT_ID ke kode ESP32
 * =====================================================
 */

// =====================================================
//  KONFIGURASI SPREADSHEET
// =====================================================

// Ganti dengan nama sheet yang diinginkan
const SHEET_LOG_NAME     = "Log Deteksi";
const SHEET_SUMMARY_NAME = "Ringkasan";
const SHEET_CONFIG_NAME  = "Konfigurasi";

// Kolom header untuk sheet log
const LOG_HEADERS = [
  "No",
  "Tanggal",
  "Interval Mulai",
  "Interval Selesai",
  "Perangkat",
  "Lokasi",
  "Jumlah Deteksi",
  "Durasi Aktif (detik)",
  "WiFi RSSI (dBm)",
  "Kualitas Sinyal"
];

// =====================================================
//  INISIALISASI SPREADSHEET
// =====================================================

function initSpreadsheet() {
  const ss = SpreadsheetApp.getActiveSpreadsheet();
  
  // Sheet Log Deteksi
  let logSheet = ss.getSheetByName(SHEET_LOG_NAME);
  if (!logSheet) {
    logSheet = ss.insertSheet(SHEET_LOG_NAME);
    setupLogSheet(logSheet);
  }
  
  // Sheet Ringkasan
  let summarySheet = ss.getSheetByName(SHEET_SUMMARY_NAME);
  if (!summarySheet) {
    summarySheet = ss.insertSheet(SHEET_SUMMARY_NAME);
    setupSummarySheet(summarySheet);
  }
  
  return { logSheet, summarySheet };
}

function setupLogSheet(sheet) {
  // Header row
  const headerRange = sheet.getRange(1, 1, 1, LOG_HEADERS.length);
  headerRange.setValues([LOG_HEADERS]);
  
  // Format header
  headerRange.setBackground("#1565C0");
  headerRange.setFontColor("#FFFFFF");
  headerRange.setFontWeight("bold");
  headerRange.setHorizontalAlignment("center");
  headerRange.setFontSize(11);
  
  // Freeze header row
  sheet.setFrozenRows(1);
  
  // Set lebar kolom
  sheet.setColumnWidth(1, 50);   // No
  sheet.setColumnWidth(2, 170);  // Timestamp
  sheet.setColumnWidth(3, 100);  // Tanggal
  sheet.setColumnWidth(4, 80);   // Jam
  sheet.setColumnWidth(5, 120);  // Perangkat
  sheet.setColumnWidth(6, 120);  // Lokasi
  sheet.setColumnWidth(7, 140);  // Status
  sheet.setColumnWidth(8, 130);  // Jumlah Deteksi
  sheet.setColumnWidth(9, 130);  // RSSI
  sheet.setColumnWidth(10, 130); // Kualitas Sinyal
}

function setupSummarySheet(sheet) {
  sheet.getRange("A1").setValue("📊 RINGKASAN DETEKSI PIR");
  sheet.getRange("A1").setFontSize(16).setFontWeight("bold").setFontColor("#1565C0");
  
  sheet.getRange("A3").setValue("Total Deteksi Hari Ini:");
  sheet.getRange("A4").setValue("Total Semua Waktu:");
  sheet.getRange("A5").setValue("Perangkat Aktif:");
  sheet.getRange("A6").setValue("Terakhir Diperbarui:");
  
  sheet.getRange("A3:A6").setFontWeight("bold");
  sheet.setColumnWidth(1, 200);
  sheet.setColumnWidth(2, 200);
}

// =====================================================
//  FUNGSI UTAMA - TERIMA DATA (HTTP POST)
// =====================================================

function doGet(e) {
  try {
    if (e.parameter && e.parameter.device) {
      // Ada data dari ESP32
      const data = {
        device    : e.parameter.device    || "",
        location  : e.parameter.location  || "",
        tsStart  : e.parameter.tsStart  || "",
        tsEnd    : e.parameter.tsEnd    || "",
        date     : e.parameter.date     || "",
        count     : parseInt(e.parameter.count) || 0,
        duration : parseFloat(e.parameter.duration)  || 0.0,
        rssi     : parseInt(e.parameter.rssi)        || 0
      };
      const result = saveData(data);
      return ContentService
        .createTextOutput(JSON.stringify({ status: "success", row: result.rowNumber }))
        .setMimeType(ContentService.MimeType.JSON);
    }
    // Cek koneksi biasa
    return ContentService
      .createTextOutput(JSON.stringify({ status: "online" }))
      .setMimeType(ContentService.MimeType.JSON);
  } catch (error) {
    return ContentService
      .createTextOutput(JSON.stringify({ status: "error", message: error.toString() }))
      .setMimeType(ContentService.MimeType.JSON);
  }
}

// =====================================================
//  SIMPAN DATA KE SPREADSHEET
// =====================================================

function saveData(data) {
  const { logSheet } = initSpreadsheet();
  
  // Tentukan kualitas sinyal WiFi berdasarkan RSSI
  const rssiQuality = getRSSIQuality(data.rssi || 0);
  
  // Nomor urut baris (hitung data yang sudah ada)
  const lastRow    = logSheet.getLastRow();
  const rowNumber  = lastRow;  // Header di baris 1
  const newRowNum  = lastRow + 1;
  
  // Data baris baru
  const rowData = [
    rowNumber,
    data.date,
    data.tsStart,
    data.tsEnd,
    data.device,
    data.location,
    data.count,
    data.duration,
    data.rssi,
    rssiQuality
  ];
  
  // Tulis ke baris baru
  logSheet.getRange(newRowNum, 1, 1, rowData.length).setValues([rowData]);
  
  // Warnai baris berdasarkan status
  colorizeRow(logSheet, newRowNum, data.status);
  
  // Update ringkasan
  updateSummary(data);
  
  Logger.log("Data disimpan di baris: " + newRowNum);
  return { rowNumber: newRowNum };
}

// =====================================================
//  WARNAI BARIS BERDASARKAN STATUS
// =====================================================

function colorizeRow(sheet, rowNum, status) {
  const range = sheet.getRange(rowNum, 1, 1, LOG_HEADERS.length);
  
  let bgColor = "#FFFFFF";
  let fontColor = "#000000";
  
  switch(status) {
    case "MOTION_DETECTED":
      bgColor   = "#FFF9C4";  // Kuning muda
      fontColor = "#E65100";  // Oranye tua
      break;
    case "SYSTEM_START":
      bgColor   = "#E8F5E9";  // Hijau muda
      fontColor = "#1B5E20";  // Hijau tua
      break;
    case "ONLINE":
      bgColor   = "#E3F2FD";  // Biru muda
      fontColor = "#0D47A1";  // Biru tua
      break;
    default:
      bgColor = "#FAFAFA";
  }
  
  range.setBackground(bgColor);
  range.setFontColor(fontColor);
  
  // Highlight kolom status lebih terang
  if (status === "MOTION_DETECTED") {
    sheet.getRange(rowNum, 7).setBackground("#FF6F00").setFontColor("#FFFFFF").setFontWeight("bold");
  }
}

// =====================================================
//  KUALITAS SINYAL WiFi
// =====================================================

function getRSSIQuality(rssi) {
  if (rssi >= -50) return "Sangat Baik";
  if (rssi >= -60) return "Baik";
  if (rssi >= -70) return "Cukup";
  if (rssi >= -80) return "Lemah";
  return "Sangat Lemah";
}

// =====================================================
//  UPDATE RINGKASAN
// =====================================================

function updateSummary(data) {
  const ss = SpreadsheetApp.getActiveSpreadsheet();
  const summarySheet = ss.getSheetByName(SHEET_SUMMARY_NAME);
  if (!summarySheet) return;
  
  const today = new Date().toLocaleDateString("id-ID");
  
  // Hitung deteksi hari ini
  const logSheet = ss.getSheetByName(SHEET_LOG_NAME);
  let todayCount = 0;
  let totalCount = 0;
  let devices = new Set();
  
  if (logSheet) {
    const lastRow = logSheet.getLastRow();
    if (lastRow > 1) {
      const logData = logSheet.getRange(2, 1, lastRow - 1, 10).getValues();
      for (const row of logData) {
        if (row[6] === "MOTION_DETECTED") {
          totalCount++;
          if (row[2] === new Date().toLocaleDateString("id-ID", {
            year: "numeric", month: "2-digit", day: "2-digit"
          }).split("/").reverse().join("-")) {
            todayCount++;
          }
          devices.add(row[4]); // Tambah nama perangkat
        }
      }
    }
  }
  
  summarySheet.getRange("B3").setValue(data.count || 0);
  summarySheet.getRange("B4").setValue(totalCount);
  summarySheet.getRange("B5").setValue(devices.size || 1);
  summarySheet.getRange("B6").setValue(new Date().toLocaleString("id-ID"));
  
  summarySheet.getRange("B3:B6").setFontWeight("bold").setFontColor("#1565C0");
}

// =====================================================
//  FUNGSI UTILITAS - BUAT TRIGGER OTOMATIS (Opsional)
// =====================================================

/**
 * Jalankan fungsi ini SEKALI dari Apps Script Editor
 * untuk membuat trigger pembersihan data otomatis setiap minggu
 */
function createWeeklyCleanupTrigger() {
  ScriptApp.newTrigger("weeklyCleanup")
    .timeBased()
    .everyWeeks(1)
    .create();
  Logger.log("Trigger mingguan berhasil dibuat.");
}

/**
 * Fungsi untuk mengarsipkan data lama (lebih dari 30 hari)
 * ke sheet terpisah agar spreadsheet tidak terlalu berat
 */
function weeklyCleanup() {
  const ss = SpreadsheetApp.getActiveSpreadsheet();
  const logSheet = ss.getSheetByName(SHEET_LOG_NAME);
  if (!logSheet) return;
  
  const cutoffDate = new Date();
  cutoffDate.setDate(cutoffDate.getDate() - 30);  // 30 hari lalu
  
  Logger.log("Cleanup: Memindahkan data sebelum " + cutoffDate.toLocaleDateString("id-ID"));
  
  // Arsip sheet
  let archiveName = "Arsip " + cutoffDate.toLocaleDateString("id-ID");
  let archiveSheet = ss.getSheetByName(archiveName);
  if (!archiveSheet) {
    archiveSheet = ss.insertSheet(archiveName);
    setupLogSheet(archiveSheet);
  }
  
  // Pindahkan baris lama
  const lastRow = logSheet.getLastRow();
  let rowsToDelete = [];
  
  for (let i = 2; i <= lastRow; i++) {
    const rowTimestamp = logSheet.getRange(i, 2).getValue();
    const rowDate = new Date(rowTimestamp);
    
    if (rowDate < cutoffDate) {
      const rowData = logSheet.getRange(i, 1, 1, LOG_HEADERS.length).getValues();
      archiveSheet.appendRow(rowData[0]);
      rowsToDelete.push(i);
    }
  }
  
  // Hapus dari sheet utama (dari bawah ke atas agar index tidak bergeser)
  for (let i = rowsToDelete.length - 1; i >= 0; i--) {
    logSheet.deleteRow(rowsToDelete[i]);
  }
  
  Logger.log("Cleanup selesai: " + rowsToDelete.length + " baris diarsipkan.");
}

// =====================================================
//  FUNGSI TEST - Jalankan dari editor untuk testing
// =====================================================

function testSaveData() {
  const testData = {
    device    : "ESP32-PIR-TEST",
    location  : "Prototipe sawah",
    timestamp : new Date().toLocaleString("id-ID"),
    date      : new Date().toLocaleDateString("id-ID"),
    time      : new Date().toLocaleTimeString("id-ID"),
    status    : "MOTION_DETECTED",
    count     : 1,
    rssi      : -55
  };
  
  const result = saveData(testData);
  Logger.log("Test berhasil: baris " + result.rowNumber);
  SpreadsheetApp.getUi().alert("✓ Test berhasil! Data disimpan di baris " + result.rowNumber);
}
